# Limpieza y conversión del Excel de alertamientos → formato de la app Mesa de Servicio

Este notebook toma el Excel de **Seguimiento de Alertamientos y Gestión de Riesgos** (el archivo original,
o uno con la misma información) y genera el archivo que usa la app de Power Apps, con los datos limpios:

| Entra | Sale |
|---|---|
| Excel original (hoja con DIA, MES, AÑO, ANALISTA…, ESTADO…) | `Seguimiento_de_Alertamientos_y_Gestion_de_Riesgos.xlsx` con **las mismas hojas, tablas y columnas que espera la app** |
| (opcional) el Excel que la app usa hoy | Se conservan sus **USUARIOS, CONFIG y correos** reales |
| | `Reporte_limpieza_datos.xlsx`: qué se corrigió, qué se unificó y qué hay que revisar a mano |

**Qué limpia**
- Espacios de más, espacios no separables, saltos de línea sueltos y filas vacías.
- Nombres en MAYÚSCULAS o minúsculas → formato *Nombre Propio* (respetando siglas como TI, PQR, PLAFT, ACH).
- Variantes del mismo valor (`sofia cadavid`, `SOFÍA CADAVID`, `Sofía  Cadavid`) → un solo valor.
  En áreas, causales y subcausales también corrige errores de **una letra** (`Fallas en Procesoss`).
  En nombres de personas solo **sugiere** (para no mezclar a *Andrea* con *Andrés*).
- Meses escritos como número, abreviatura o en inglés (`2`, `feb`, `February`) → `Febrero`.
- Fechas escritas de cualquier forma (fecha de Excel, `12/01/2024`, `2024-01-12`) → DIA / MES / AÑO.
- Estados (`EN SEGUIMIENTO`, `solucionado`, `cerrada`…) → los 5 estados de la app.
- Respuesta de causa raíz (`si`, `SI`, `x`, `parcial`…) → `Sí` / `No` / `Parcialmente` / `En evaluación`.
- IDs repetidos o vacíos → se asigna un IdNum nuevo (y queda en el reporte).
- Filas duplicadas exactas → se eliminan; duplicados probables → se reportan.

**Cómo usarlo:** ponga el Excel original en la misma carpeta que este notebook, revise la celda de
**PARÁMETROS** y ejecute todo (*Run All*). Funciona en Jupyter, VS Code o Google Colab
(en Colab suba primero el archivo al panel de archivos). Requiere `pandas` y `openpyxl`.

> ⚠️ El notebook **nunca modifica** el archivo de origen: siempre escribe un archivo nuevo.

## 1. Parámetros
Es la única celda que normalmente hay que tocar.

In [ ]:
from pathlib import Path

# Archivo de entrada (el Excel original con los casos)
RUTA_ORIGEN = Path("Seguimiento_de_Alertamientos_y_Gestion_de_Riesgos_ORIGINAL.xlsx")

# OPCIONAL: el Excel que la app usa hoy (descárguelo de OneDrive). Si lo indica, se conservan sus
# USUARIOS, CONFIG y los correos reales de los analistas. Déjelo en None si no aplica.
RUTA_APP_ACTUAL = None   # ej.: Path("Seguimiento_de_Alertamientos_y_Gestion_de_Riesgos_APP.xlsx")

# Archivos de salida
RUTA_SALIDA = Path("Seguimiento_de_Alertamientos_y_Gestion_de_Riesgos.xlsx")
RUTA_REPORTE = Path("Reporte_limpieza_datos.xlsx")

# Administrador de la app (si se deja vacío y no hay RUTA_APP_ACTUAL, se crea uno de ejemplo)
ADMIN_CORREO = ""        # ej.: "jsrueda@poligran.edu.co"
ADMIN_NOMBRE = ""        # ej.: "Juan Sebastián Rueda Galvis"

# Días para la fecha límite de cada caso (FECHA_LIMITE = fecha de alertamiento + DIAS_LIMITE)
DIAS_LIMITE = 60

# Correos reales de las personas (nombre tal como aparece en el Excel → correo). Opcional.
# Las personas sin correo quedan con uno de ejemplo @example.com, marcado en el reporte.
CORREOS = {
    # "Andrea Gil": "andrea.gil@empresa.com",
}

# Correcciones manuales de valores (se aplican antes de todo lo demás). Útil para las
# sugerencias que deja el reporte en la hoja SUGERENCIAS.
CORRECCIONES = {
    # "Camilo Toress": "Camilo Torres",
}

# Reglas
EXCLUIR_FILAS_CON_ERRORES = True     # filas sin fecha válida o sin descripción no pasan a la app
ELIMINAR_DUPLICADOS_EXACTOS = True   # filas idénticas en todas las columnas originales
SIMILITUD_SUGERENCIAS = 0.85         # 0–1: desde qué parecido se sugiere revisar dos valores

## 2. Estructura que espera la app
Nombres de hojas, tablas y columnas tal como los usa Power Apps. **No los cambie**: la app y los flujos
dependen de ellos (incluidos los nombres originales `CASUAL`, `SUBCASUAL` y `ANALISTA ASIGANDO`).

In [ ]:
import re
import unicodedata
from collections import Counter, OrderedDict
from datetime import date, datetime, timedelta
from difflib import SequenceMatcher

import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo

HOJA_CASOS, TABLA_CASOS = "Hoja 1", "TablaCasos"
COLUMNA_ID = "IdNum"
COLUMNAS_ORIGINALES = [
    "DIA", "MES", "AÑO", "ANALISTA ENCARGADA UAC", "SPONSOR", "ÁREA ENCARGADA",
    "ANALISTA ASIGANDO", "CASUAL", "SUBCASUAL", "RESPONSABLE OPERATIVO DEL ALERTAMIENTO",
    "DESCRIPCION DEL RIESGO", "AVANCES DEL CASO", "PLAN DE ACCION", "ESTADO",
    "MES DE CIERRE DEL CASO", "AÑO DE CIERRE DEL CASO",
    "EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ",
]
COLUMNAS_TECNICAS = [
    "FECHA_CREACION", "CORREO_CREADOR", "NOMBRE_CREADOR", "CORREO_ANALISTA_ASIGNADO",
    "FECHA_ASIGNACION", "FECHA_LIMITE", "FECHA_CIERRE", "MOTIVO_DEVOLUCION", "ESTADO_ANTERIOR",
    "FECHA_VENCIMIENTO", "FECHA_ESCALAMIENTO", "ESCALAMIENTO_ENVIADO", "ULTIMA_ACTUALIZACION",
    "RESPONSABLE_ULTIMA_ACTUALIZACION", "FECHA_DEVOLUCION", "ALERTAS_ENVIADAS",
    "NIVEL_ESCALAMIENTO", "TOKEN_CREACION", "VERSION", "ORIGEN_REGISTRO",
]
COLUMNAS_CASOS = [COLUMNA_ID] + COLUMNAS_ORIGINALES + COLUMNAS_TECNICAS
COLUMNAS_FECHA_TEXTO = ["FECHA_CREACION", "FECHA_ASIGNACION", "FECHA_LIMITE", "FECHA_CIERRE",
                        "FECHA_VENCIMIENTO", "FECHA_ESCALAMIENTO", "ULTIMA_ACTUALIZACION",
                        "FECHA_DEVOLUCION"]

COLUMNAS_HISTORIAL = ["ID_HISTORIAL", "ID_CASO", "FECHA", "TIPO_EVENTO", "ESTADO_ANTERIOR",
                      "ESTADO_NUEVO", "DETALLE", "VALOR_ANTERIOR", "VALOR_NUEVO",
                      "USUARIO_CORREO", "USUARIO_NOMBRE", "ORIGEN"]
COLUMNAS_USUARIOS = ["CORREO", "NOMBRE", "ROL", "AREA", "ACTIVO", "FECHA_ALTA", "OBSERVACIONES"]
COLS_NOTIF = ["ID_NOTIFICACION", "ID_CASO", "TIPO", "CLAVE_UNICA", "DESTINATARIOS",
              "ASUNTO", "FECHA_ENVIO", "RESULTADO", "DETALLE", "FLUJO"]
COLS_LOG = ["ID_ERROR", "FECHA", "FLUJO", "ID_EJECUCION", "ID_CASO", "ACCION",
            "MENSAJE", "DETALLE", "USUARIO"]

MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Septiembre", "Octubre", "Noviembre", "Diciembre"]
ESTADOS = [  # CODIGO (valor en la columna ESTADO), ETIQUETA, ORDEN, COLOR, ES_FINAL
    ("Abierto", "Abierto", 1, "#1F6FEB", "No"),
    ("En Proceso", "En Proceso", 2, "#B7791F", "No"),
    ("Devuelto", "Devuelto", 3, "#C2410C", "No"),
    ("Cerrado", "Cerrado / Solucionado", 4, "#1E7D3A", "Sí"),
    ("Vencido", "Vencido", 5, "#B91C1C", "No"),
]
ESTADOS_VALIDOS = [e[0] for e in ESTADOS]
VALORES_RAIZ = ["Sí", "No", "Parcialmente", "En evaluación"]
ROLES = ["ADMINISTRADOR", "ESPECIALISTA", "SOLICITANTE"]

CONFIG_DEFECTO = [
    ("NOMBRE_EMPRESA", "CHUBB", "Texto del encabezado de la app y de los correos."),
    ("NOMBRE_APP", "Seguimiento de Alertamientos y Gestión de Riesgos", "Nombre mostrado en app y correos."),
    ("URL_APP", "[URL_APP]", "Enlace de reproducción de la app (Detalles de la app > Vínculo web). Completar tras publicar."),
    ("DIAS_LIMITE", str(DIAS_LIMITE), "Días calendario máximos por caso. FECHA_LIMITE = FECHA_CREACION + DIAS_LIMITE."),
    ("DIAS_PROXIMO_VENCER", "15", "Días restantes desde los que un caso se considera 'próximo a vencer' en el dashboard."),
    ("UMBRALES_ALERTA", "15;7;3;1", "Días restantes en que se envía alerta preventiva (separados por ';')."),
    ("ID_INICIAL", "1", "Primer IdNum si la tabla está vacía. Con datos, el siguiente IdNum = máximo existente + 1."),
    ("ZONA_HORARIA", "SA Pacific Standard Time", "Zona horaria Windows usada por los flujos (Bogotá, UTC-5)."),
    ("CORREO_ESCALAMIENTO_NIVEL_1", "[CORREO_ESCALAMIENTO_NIVEL_1]", "Destinatario(s) del escalamiento inicial (separar con ';'). COMPLETAR."),
    ("CORREO_ESCALAMIENTO_NIVEL_2", "[CORREO_ESCALAMIENTO_NIVEL_2]", "Escalamiento nivel 2 (preparado, inactivo). COMPLETAR."),
    ("CORREO_ESCALAMIENTO_NIVEL_3", "[CORREO_ESCALAMIENTO_NIVEL_3]", "Escalamiento nivel 3 (preparado, inactivo). COMPLETAR."),
    ("CORREO_ADICIONAL", "[CORREO_ADICIONAL]", "Copia adicional en escalamientos. COMPLETAR o dejar el marcador."),
    ("ESCALAMIENTO_NIVELES_ACTIVO", "No", "Sí = activa escalamientos nivel 2 y 3. Dejar en No hasta definir la política."),
    ("DIAS_ESCALAMIENTO_NIVEL_2", "15", "Días de vencido para nivel 2 (solo si ESCALAMIENTO_NIVELES_ACTIVO = Sí)."),
    ("DIAS_ESCALAMIENTO_NIVEL_3", "30", "Días de vencido para nivel 3 (solo si ESCALAMIENTO_NIVELES_ACTIVO = Sí)."),
    ("MODO_PRUEBAS", "Sí", "Sí = todos los correos se redirigen a CORREO_PRUEBAS. Cambiar a No en producción."),
    ("CORREO_PRUEBAS", "[CORREO_PRUEBAS]", "Buzón que recibe todos los correos mientras MODO_PRUEBAS = Sí. COMPLETAR."),
    ("BUZON_COMPARTIDO", "", "Opcional: buzón compartido remitente (requiere permiso Enviar como). Vacío = cuenta del flujo."),
    ("DOMINIOS_PERMITIDOS", "", "Opcional: dominios corporativos permitidos separados por ';'. Vacío = no valida dominio."),
    ("RUTA_PLANTILLAS", "/MesaServicio/PlantillasCorreos", "Carpeta (OneDrive/SharePoint) con las plantillas HTML."),
]
PLANTILLAS = [
    ("NUEVO_CASO", "01_nuevo_caso_asignado.html", "[Mesa de Servicio] Nuevo caso asignado #{{ID_CASO}} – {{CAUSAL}}", "Sí"),
    ("REASIGNACION", "02_reasignacion_caso.html", "[Mesa de Servicio] Caso #{{ID_CASO}} reasignado a usted", "Sí"),
    ("DEVOLUCION", "03_caso_devuelto.html", "[Mesa de Servicio] Su caso #{{ID_CASO}} fue devuelto – se requiere información", "Sí"),
    ("SOLUCIONADO", "04_caso_solucionado.html", "[Mesa de Servicio] Caso #{{ID_CASO}} solucionado y cerrado", "Sí"),
    ("PROXIMO_VENCER", "05_caso_proximo_vencer.html", "[Mesa de Servicio] Caso #{{ID_CASO}} vence en {{DIAS_RESTANTES}} día(s)", "Sí"),
    ("VENCIDO", "06_caso_vencido.html", "[Mesa de Servicio] Caso #{{ID_CASO}} VENCIDO", "Sí"),
    ("ESCALAMIENTO", "07_escalamiento.html", "[ESCALAMIENTO N{{NIVEL}}] Caso #{{ID_CASO}} vencido hace {{DIAS_VENCIDO}} día(s) – {{AREA}}", "Sí"),
    ("RESPUESTA_DEVOLUCION", "08_respuesta_devolucion.html", "[Mesa de Servicio] El solicitante respondió la devolución del caso #{{ID_CASO}}", "Sí"),
]
EQUIVALENCIAS_ESTADO_HOJA = [
    ("En Proceso", "En Proceso"), ("EN PROCESO", "En Proceso"), ("En Seguimiento", "En Proceso"),
    ("EN SEGUIMIENTO", "En Proceso"), ("Seguimiento", "En Proceso"), ("Cerrado", "Cerrado"),
    ("CERRADO", "Cerrado"), ("Solucionado", "Cerrado"), ("", "Abierto"),
]

assert RUTA_ORIGEN.exists(), f"No encuentro el archivo de origen: {RUTA_ORIGEN.resolve()}"
assert RUTA_SALIDA.resolve() != RUTA_ORIGEN.resolve(), "RUTA_SALIDA no puede ser el mismo archivo de origen."
if RUTA_APP_ACTUAL is not None:
    RUTA_APP_ACTUAL = Path(RUTA_APP_ACTUAL)
    assert RUTA_APP_ACTUAL.exists(), f"No encuentro RUTA_APP_ACTUAL: {RUTA_APP_ACTUAL.resolve()}"
print("Estructura cargada. Origen:", RUTA_ORIGEN.name)

## 3. Funciones de limpieza
Todas las correcciones quedan registradas en `CAMBIOS` y los problemas en `PROBLEMAS`; al final
se guardan en el reporte.

In [ ]:
CAMBIOS = []      # (fila_origen, IdNum, columna, antes, despues, regla)
PROBLEMAS = []    # (nivel, fila_origen, IdNum, columna, valor, mensaje)


def registrar_cambio(fila, id_, col, antes, despues, regla):
    if str(antes if antes is not None else "") != str(despues if despues is not None else ""):
        CAMBIOS.append((fila, id_, col, antes, despues, regla))


def registrar_problema(nivel, fila, id_, col, valor, mensaje):
    PROBLEMAS.append((nivel, fila, id_, col, valor, mensaje))


_ESPACIOS = re.compile(r"[ \t   　]+")
_INVISIBLES = dict.fromkeys(map(ord, "​‌‍﻿"), None)


def texto(v, multilinea=False):
    """Texto limpio: sin espacios dobles ni invisibles. Números enteros sin '.0'."""
    if v is None:
        return ""
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    if isinstance(v, datetime):
        v = v.strftime("%Y-%m-%d %H:%M:%S") if (v.hour or v.minute or v.second) else v.strftime("%Y-%m-%d")
    elif isinstance(v, date):
        v = v.strftime("%Y-%m-%d")
    s = unicodedata.normalize("NFC", str(v)).translate(_INVISIBLES)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    if multilinea:
        s = "\n".join(_ESPACIOS.sub(" ", linea).strip() for linea in s.split("\n"))
        s = re.sub(r"\n{3,}", "\n\n", s).strip()
    else:
        s = _ESPACIOS.sub(" ", s.replace("\n", " ")).strip()
    return s


def clave(v):
    """Clave de comparación: sin tildes, sin mayúsculas, sin espacios ni signos."""
    s = unicodedata.normalize("NFKD", texto(v)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "", s)


def vacio(v):
    return texto(v) in ("", "-", "--", "n/a", "N/A", "NA", "na", "null", "None", "nan")


SIGLAS = {"TI", "IT", "PQR", "PQRS", "PLAFT", "SARLAFT", "SARO", "ACH", "API", "UAC", "IFRS", "OTP",
          "SMS", "WAF", "CORE", "ATM", "SFC", "DIAN", "UIAF", "PSE", "CRM", "ERP", "SAP", "VPN", "BD"}
PARTICULAS = {"de", "del", "la", "las", "los", "y", "e", "o", "u", "a", "en", "el", "al", "para",
              "por", "con", "sin", "da", "do", "dos", "van", "von"}


def es_mixto(s):
    letras = [c for c in s if c.isalpha()]
    return any(c.isupper() for c in letras) and any(c.islower() for c in letras)


def formato(s, modo="titulo"):
    """Solo actúa si el texto está TODO en mayúsculas o todo en minúsculas (si está mezclado se respeta).
    modo 'titulo': Cada Palabra Con Mayúscula (nombres, áreas, causales).
    modo 'oracion': Solo la primera letra (subcausales)."""
    if not s or es_mixto(s) or not any(c.isalpha() for c in s):
        return s
    palabras = s.lower().split(" ")
    salida = []
    for i, p in enumerate(palabras):
        base = re.sub(r"[^\w]", "", p).upper()
        if base in SIGLAS:
            salida.append(p.upper())
        elif modo == "titulo" and (i == 0 or p not in PARTICULAS):
            salida.append(p[:1].upper() + p[1:])
        elif modo == "oracion" and i == 0:
            salida.append(p[:1].upper() + p[1:])
        else:
            salida.append(p)
    return " ".join(salida)


def distancia(a, b, maximo=2):
    """Distancia de edición (Levenshtein) con corte temprano."""
    if abs(len(a) - len(b)) > maximo:
        return maximo + 1
    previo = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        actual = [i]
        for j, cb in enumerate(b, 1):
            actual.append(min(previo[j] + 1, actual[j - 1] + 1, previo[j - 1] + (ca != cb)))
        if min(actual) > maximo:
            return maximo + 1
        previo = actual
    return previo[-1]


# ---- Meses, años, fechas -------------------------------------------------------------
_MES_ALIAS = {}
for _i, _m in enumerate(MESES, 1):
    _MES_ALIAS[clave(_m)] = _i
    _MES_ALIAS[clave(_m)[:3]] = _i
for _k, _i in {"set": 9, "sept": 9, "setiembre": 9, "jan": 1, "january": 1, "february": 2, "march": 3,
               "apr": 4, "april": 4, "june": 6, "july": 7, "aug": 8, "august": 8, "september": 9,
               "october": 10, "november": 11, "dec": 12, "december": 12}.items():
    _MES_ALIAS[_k] = _i


def mes_num(v):
    if isinstance(v, (datetime, date)):
        return v.month
    s = texto(v)
    if re.fullmatch(r"\d{1,2}", s) and 1 <= int(s) <= 12:
        return int(s)
    return _MES_ALIAS.get(clave(s))


def anio_num(v):
    if isinstance(v, (datetime, date)):
        return v.year
    s = texto(v)
    if re.fullmatch(r"\d{2}", s):
        return 2000 + int(s)
    if re.fullmatch(r"\d{4}", s) and 1990 <= int(s) <= 2100:
        return int(s)
    return None


def dia_num(v):
    if isinstance(v, (datetime, date)):
        return v.day
    s = texto(v)
    return int(s) if re.fullmatch(r"\d{1,2}", s) and 1 <= int(s) <= 31 else None


def a_fecha(v):
    """Convierte fechas de Excel, números de serie y textos comunes a date (o None)."""
    if v is None or vacio(v):
        return None
    if isinstance(v, datetime):
        return v.date()
    if isinstance(v, date):
        return v
    if isinstance(v, (int, float)) and 20000 < v < 80000:            # número de serie de Excel
        return (datetime(1899, 12, 30) + timedelta(days=float(v))).date()
    s = texto(v)
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y",
                "%d/%m/%y", "%d.%m.%Y", "%Y/%m/%d", "%d/%m/%Y %H:%M", "%d/%m/%Y %H:%M:%S"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    m = re.fullmatch(r"(\d{1,2})\s*(?:de\s+)?([A-Za-zÁÉÍÓÚáéíóú]+)\.?\s*(?:de\s+|del\s+)?(\d{2,4})", s)
    if m and mes_num(m.group(2)) and anio_num(m.group(3)):
        try:
            return date(anio_num(m.group(3)), mes_num(m.group(2)), int(m.group(1)))
        except ValueError:
            return None
    return None


def a_fecha_hora_texto(v, solo_fecha=False):
    """Texto 'aaaa-mm-dd hh:mm:ss' (o 'aaaa-mm-dd') que es como la app guarda las fechas técnicas."""
    if v is None or vacio(v):
        return ""
    if isinstance(v, datetime):
        dt = v
    else:
        s = texto(v)
        dt = None
        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%dT%H:%M:%S", "%m/%d/%Y %I:%M:%S %p",
                    "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M"):
            try:
                dt = datetime.strptime(s[:19] if "T" in s else s, fmt)
                break
            except ValueError:
                pass
        if dt is None:
            d = a_fecha(v)
            if d is None:
                return None          # no se pudo interpretar
            dt = datetime(d.year, d.month, d.day)
    return dt.strftime("%Y-%m-%d") if solo_fecha else dt.strftime("%Y-%m-%d %H:%M:%S")


# ---- Estados y causa raíz --------------------------------------------------------------
EQUIV_ESTADO = {
    "abierto": "Abierto", "abierta": "Abierto", "nuevo": "Abierto", "nueva": "Abierto",
    "pendiente": "Abierto", "sinasignar": "Abierto", "registrado": "Abierto", "creado": "Abierto",
    "enproceso": "En Proceso", "proceso": "En Proceso", "enseguimiento": "En Proceso",
    "seguimiento": "En Proceso", "engestion": "En Proceso", "gestion": "En Proceso",
    "encurso": "En Proceso", "enprogreso": "En Proceso", "entramite": "En Proceso",
    "asignado": "En Proceso", "abiertoenproceso": "En Proceso", "enrevision": "En Proceso",
    "devuelto": "Devuelto", "devuelta": "Devuelto",
    "cerrado": "Cerrado", "cerrada": "Cerrado", "solucionado": "Cerrado", "solucionada": "Cerrado",
    "resuelto": "Cerrado", "resuelta": "Cerrado", "finalizado": "Cerrado", "finalizada": "Cerrado",
    "terminado": "Cerrado", "terminada": "Cerrado", "cerradosolucionado": "Cerrado", "completado": "Cerrado",
    "vencido": "Vencido", "vencida": "Vencido",
}
EQUIV_RAIZ = {
    "si": "Sí", "s": "Sí", "yes": "Sí", "y": "Sí", "x": "Sí", "true": "Sí", "verdadero": "Sí", "1": "Sí",
    "no": "No", "n": "No", "false": "No", "falso": "No", "0": "No",
    "parcialmente": "Parcialmente", "parcial": "Parcialmente", "enparte": "Parcialmente",
    "": "En evaluación", "enevaluacion": "En evaluación", "evaluacion": "En evaluación",
    "pendiente": "En evaluación", "porevaluar": "En evaluación", "nd": "En evaluación",
    "noaplica": "En evaluación", "na": "En evaluación",
}

# ---- Lectura de hojas --------------------------------------------------------------------
ALIAS_COLUMNAS = {
    "IdNum": ["idnum", "id", "idcaso", "nocaso", "numerocaso", "numero", "consecutivo", "caso"],
    "DIA": ["dia"], "MES": ["mes"], "AÑO": ["ano", "anio", "year"],
    "ANALISTA ENCARGADA UAC": ["analistaencargadauac", "analistauac", "analistaencargado", "analistaencargada"],
    "SPONSOR": ["sponsor", "patrocinador"],
    "ÁREA ENCARGADA": ["areaencargada", "area"],
    "ANALISTA ASIGANDO": ["analistaasigando", "analistaasignado", "analistaasignada"],
    "CASUAL": ["casual", "causal", "causa"],
    "SUBCASUAL": ["subcasual", "subcausal", "subcausa"],
    "RESPONSABLE OPERATIVO DEL ALERTAMIENTO": ["responsableoperativodelalertamiento", "responsableoperativo", "responsable"],
    "DESCRIPCION DEL RIESGO": ["descripciondelriesgo", "descripcion", "riesgo"],
    "AVANCES DEL CASO": ["avancesdelcaso", "avances"],
    "PLAN DE ACCION": ["plandeaccion", "plan"],
    "ESTADO": ["estado"],
    "MES DE CIERRE DEL CASO": ["mesdecierredelcaso", "mesdecierre", "mescierre"],
    "AÑO DE CIERRE DEL CASO": ["anodecierredelcaso", "anodecierre", "aniodecierre", "anocierre"],
    "EL PLAN DE ACCION SOLUCIONO EL PROBLEMA DE RAIZ": [],  # se llena abajo
    "FECHA": ["fecha", "fechaalertamiento", "fechadealertamiento", "fechadelalertamiento", "fecharegistro"],
    "FECHA DE CIERRE": ["fechadecierre", "fechacierre"],
}
ALIAS_COLUMNAS["EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ"] = [
    "elplandeaccionsolucionoelproblemaderaiz", "solucionoelproblemaderaiz", "solucionoraiz",
    "problemaderaiz", "causaraiz", "raiz"]
del ALIAS_COLUMNAS["EL PLAN DE ACCION SOLUCIONO EL PROBLEMA DE RAIZ"]
for _c in COLUMNAS_TECNICAS:
    ALIAS_COLUMNAS[_c] = [clave(_c)]
_ALIAS_A_COLUMNA = {a: col for col, alias in ALIAS_COLUMNAS.items() for a in alias}


def leer_hoja(ws, columnas_esperadas=None, minimo=3):
    """Busca la fila de encabezados en las primeras 30 filas y devuelve (encabezados, filas[dict], fila_inicio)."""
    filas = list(ws.iter_rows(values_only=True))
    mejor = (0, None)
    for i, fila in enumerate(filas[:30]):
        claves = [clave(v) for v in fila]
        if columnas_esperadas:
            aciertos = sum(1 for c in columnas_esperadas if clave(c) in claves)
        else:
            aciertos = sum(1 for c in claves if c in _ALIAS_A_COLUMNA)
        if aciertos > mejor[0]:
            mejor = (aciertos, i)
    if mejor[1] is None or mejor[0] < minimo:
        return None, [], None
    i = mejor[1]
    encabezados = [texto(v) for v in filas[i]]
    datos = []
    for n, fila in enumerate(filas[i + 1:], start=i + 2):
        if all(v is None or texto(v) == "" for v in fila):
            continue
        datos.append({"__fila__": n, **{encabezados[j]: fila[j] for j in range(min(len(fila), len(encabezados))) if encabezados[j]}})
    return encabezados, datos, i + 1


def leer_tabla_simple(wb, hoja, columnas):
    """Lee hojas auxiliares (USUARIOS, CAT_*, CONFIG, HISTORIAL…) si existen; devuelve lista de dict."""
    if wb is None or hoja not in wb.sheetnames:
        return []
    enc, datos, _ = leer_hoja(wb[hoja], columnas, minimo=1)
    if enc is None:
        return []
    mapa = {}
    for e in enc:
        for c in columnas:
            if clave(e) == clave(c):
                mapa[e] = c
    return [{mapa[k]: v for k, v in d.items() if k in mapa} for d in datos]


print("Funciones listas.")

## 4. Leer el archivo de origen
Se busca automáticamente la hoja y la fila de encabezados con los casos, y se reconocen las columnas
aunque estén escritas distinto (`CAUSAL`/`CASUAL`, `ANALISTA ASIGNADO`/`ASIGANDO`, con o sin tildes).

In [ ]:
wb_origen = load_workbook(RUTA_ORIGEN, data_only=True)
wb_app = load_workbook(RUTA_APP_ACTUAL, data_only=True) if RUTA_APP_ACTUAL else None

mejor_hoja = None
for ws in wb_origen.worksheets:
    enc, datos, fila_enc = leer_hoja(ws)
    if enc is None:
        continue
    reconocidas = sum(1 for e in enc if clave(e) in _ALIAS_A_COLUMNA)
    if mejor_hoja is None or reconocidas > mejor_hoja[0]:
        mejor_hoja = (reconocidas, ws.title, enc, datos, fila_enc)
assert mejor_hoja, "No encontré ninguna hoja con encabezados de casos (DIA, MES, ESTADO, DESCRIPCION…)."
_, HOJA_ORIGEN, ENCABEZADOS, FILAS_CRUDAS, FILA_ENCABEZADO = mejor_hoja

MAPA_COLUMNAS, NO_RECONOCIDAS = {}, []
for e in ENCABEZADOS:
    if not e:
        continue
    destino = _ALIAS_A_COLUMNA.get(clave(e))
    if destino and destino not in MAPA_COLUMNAS.values():
        MAPA_COLUMNAS[e] = destino
    else:
        NO_RECONOCIDAS.append(e)

FILAS = []
for d in FILAS_CRUDAS:
    f = {"__fila__": d["__fila__"]}
    for k, v in d.items():
        if k in MAPA_COLUMNAS:
            f[MAPA_COLUMNAS[k]] = v
    FILAS.append(f)

faltantes = [c for c in [COLUMNA_ID] + COLUMNAS_ORIGINALES if c not in MAPA_COLUMNAS.values()]
tiene_fecha_unica = "FECHA" in MAPA_COLUMNAS.values()
print(f"Hoja de casos: '{HOJA_ORIGEN}' (encabezados en la fila {FILA_ENCABEZADO}), {len(FILAS)} filas con datos.")
print("Columnas reconocidas:", len(MAPA_COLUMNAS))
for e, c in MAPA_COLUMNAS.items():
    if e != c:
        print(f"   '{e}'  →  '{c}'")
if NO_RECONOCIDAS:
    print("Columnas que se ignoran (no existen en la app):", NO_RECONOCIDAS)
if faltantes:
    print("Columnas que no vienen en el origen (se completan):", faltantes)
criticas = [c for c in ["DESCRIPCION DEL RIESGO", "ESTADO"] if c in faltantes]
if {"DIA", "MES", "AÑO"} & set(faltantes) and not tiene_fecha_unica:
    criticas.append("DIA/MES/AÑO o FECHA")
assert not criticas, f"Faltan columnas indispensables: {criticas}"

## 5. Unificar valores repetidos con distinta escritura
Para cada columna de catálogo se agrupan las variantes del mismo valor y se elige la mejor escrita
(con tildes y mayúsculas correctas; si empatan, la más frecuente).
- **Áreas, causales y subcausales:** también se unen valores que difieren en **una letra**.
- **Personas:** las diferencias de una letra solo se **sugieren** (hoja SUGERENCIAS del reporte).

In [ ]:
CAMPOS_PERSONA = ["ANALISTA ENCARGADA UAC", "SPONSOR", "ANALISTA ASIGANDO", "RESPONSABLE OPERATIVO DEL ALERTAMIENTO"]
CAMPOS_CATALOGO = {"ÁREA ENCARGADA": "titulo", "CASUAL": "titulo", "SUBCASUAL": "oracion"}
UNIFICACIONES, SUGERENCIAS = [], []
_CORRECCIONES = {clave(k): texto(v) for k, v in CORRECCIONES.items()}


def valores_catalogo_base(campo):
    """Valores que ya existen en los catálogos del archivo de la app (si se indicó)."""
    hojas = {"ANALISTA ENCARGADA UAC": "CAT_ANALISTAS_UAC", "SPONSOR": "CAT_SPONSORS",
             "ÁREA ENCARGADA": "CAT_AREAS", "ANALISTA ASIGANDO": "CAT_ANALISTAS_ASIGNADOS",
             "CASUAL": "CAT_CASUALES", "SUBCASUAL": "CAT_SUBCASUALES",
             "RESPONSABLE OPERATIVO DEL ALERTAMIENTO": "CAT_RESP_OPERATIVO"}
    valores = []
    for wb in (wb_app, wb_origen):
        valores += [texto(r.get("NOMBRE")) for r in leer_tabla_simple(wb, hojas[campo], ["NOMBRE"]) if texto(r.get("NOMBRE"))]
    return valores


def construir_mapa(campo, modo, persona):
    crudos = [texto(f.get(campo)) for f in FILAS]
    crudos = [_CORRECCIONES.get(clave(v), v) for v in crudos]
    conteo = Counter(v for v in crudos if v)
    for v in valores_catalogo_base(campo):        # los del catálogo cuentan como referencia bien escrita
        conteo[v] += 0
    grupos = OrderedDict()
    for v, n in conteo.items():
        grupos.setdefault(clave(v), Counter())[v] += n
    total = {k: sum(g.values()) for k, g in grupos.items()}

    def puntaje(item):
        v, n = item
        return (es_mixto(v), sum(1 for c in v if ord(c) > 127), n, v)

    representante = {k: formato(max(g.items(), key=puntaje)[0], modo) for k, g in grupos.items()}
    padre = {k: k for k in grupos}
    orden = sorted(grupos, key=lambda k: (-total[k], k))
    for i, k in enumerate(orden):
        for j in orden[:i]:
            if padre[j] != j:
                continue
            d = distancia(k, j, 2)
            parecido = SequenceMatcher(None, k, j).ratio()
            if not persona and d <= 1 and min(len(k), len(j)) >= 8:
                padre[k] = j
                break
            if d <= 2 or parecido >= SIMILITUD_SUGERENCIAS:
                SUGERENCIAS.append((campo, representante[k], total[k], representante[j], total[j], round(parecido, 2)))
    mapa = {}
    for k, g in grupos.items():
        destino = representante[padre[k]]
        for v, n in g.items():
            mapa[clave(v)] = destino
            if v != destino and n:
                UNIFICACIONES.append((campo, v, destino, n, "una letra de diferencia" if padre[k] != k
                                      else "mayúsculas / tildes / espacios"))
    return mapa


MAPAS = {c: construir_mapa(c, "titulo", True) for c in CAMPOS_PERSONA if c in MAPA_COLUMNAS.values()}
MAPAS.update({c: construir_mapa(c, m, False) for c, m in CAMPOS_CATALOGO.items() if c in MAPA_COLUMNAS.values()})

df_unif = pd.DataFrame(UNIFICACIONES, columns=["COLUMNA", "VALOR_ORIGINAL", "VALOR_FINAL", "VECES", "MOTIVO"])
df_sug = pd.DataFrame(SUGERENCIAS, columns=["COLUMNA", "VALOR", "VECES", "PARECIDO_A", "VECES_2", "PARECIDO"])
print(f"Valores unificados: {len(df_unif)}  ·  Sugerencias para revisar: {len(df_sug)}")
display(df_unif) if len(df_unif) else None
display(df_sug) if len(df_sug) else None

## 6. Limpiar cada caso
Fechas, meses, estados, causa raíz, textos largos e IdNum. Se registran todos los cambios.

In [ ]:
def limpiar_fila(f):
    n = f["__fila__"]
    id_txt = texto(f.get(COLUMNA_ID))
    c = {"__fila__": n, "__errores__": []}

    def err(col, valor, msg):
        c["__errores__"].append(msg)
        registrar_problema("ERROR", n, id_txt, col, valor, msg)

    def adv(col, valor, msg):
        registrar_problema("ADVERTENCIA", n, id_txt, col, valor, msg)

    # --- IdNum (se valida unicidad después)
    c[COLUMNA_ID] = int(id_txt) if re.fullmatch(r"\d+", id_txt) and int(id_txt) > 0 else None
    if c[COLUMNA_ID] is None and id_txt:
        adv(COLUMNA_ID, id_txt, "IdNum no numérico: se asignará uno nuevo.")

    # --- Fecha de alertamiento: DIA/MES/AÑO, o columna FECHA, o FECHA_CREACION
    d, m, a = dia_num(f.get("DIA")), mes_num(f.get("MES")), anio_num(f.get("AÑO"))
    fecha = None
    if d and m and a:
        try:
            fecha = date(a, m, d)
        except ValueError:
            err("DIA", f"{f.get('DIA')}/{f.get('MES')}/{f.get('AÑO')}", "La fecha DIA/MES/AÑO no existe en el calendario.")
    for alternativa in ("FECHA", "FECHA_CREACION"):
        if fecha is None and a_fecha(f.get(alternativa)):
            fecha = a_fecha(f.get(alternativa))
            registrar_cambio(n, id_txt, "DIA/MES/AÑO", f"{texto(f.get('DIA'))}/{texto(f.get('MES'))}/{texto(f.get('AÑO'))}",
                             fecha.strftime("%d/%m/%Y"), f"tomada de la columna {alternativa}")
    if fecha is None and not c["__errores__"]:
        err("DIA/MES/AÑO", f"{texto(f.get('DIA'))}/{texto(f.get('MES'))}/{texto(f.get('AÑO'))}",
            "Sin fecha de alertamiento válida.")
    if fecha and fecha > date.today():
        adv("DIA/MES/AÑO", fecha.isoformat(), "La fecha de alertamiento está en el futuro.")
    c["__fecha__"] = fecha
    if fecha:
        c["DIA"], c["MES"], c["AÑO"] = fecha.day, MESES[fecha.month - 1], fecha.year
        registrar_cambio(n, id_txt, "MES", texto(f.get("MES")), c["MES"], "mes normalizado")
    else:
        c["DIA"], c["MES"], c["AÑO"] = f.get("DIA"), texto(f.get("MES")), f.get("AÑO")

    # --- Personas y catálogos
    for col in CAMPOS_PERSONA + list(CAMPOS_CATALOGO):
        original = texto(f.get(col))
        corregido = _CORRECCIONES.get(clave(original), original)
        valor = MAPAS[col].get(clave(corregido), corregido) if col in MAPAS and corregido else corregido
        c[col] = valor
        registrar_cambio(n, id_txt, col, f.get(col), valor, "unificación / formato")
        if not valor and col != "SPONSOR":
            adv(col, "", "Valor vacío: el caso no aparecerá en los filtros de esta columna.")

    # --- Textos largos
    for col in ["DESCRIPCION DEL RIESGO", "AVANCES DEL CASO", "PLAN DE ACCION"]:
        c[col] = texto(f.get(col), multilinea=True)
        if f.get(col) is not None and c[col] != str(f.get(col)):
            registrar_cambio(n, id_txt, col, "(texto)", "(texto sin espacios sobrantes)", "espacios")
    if not c["DESCRIPCION DEL RIESGO"]:
        err("DESCRIPCION DEL RIESGO", "", "Caso sin descripción del riesgo.")

    # --- Mes / año de cierre
    mc_raw, ac_raw = f.get("MES DE CIERRE DEL CASO"), f.get("AÑO DE CIERRE DEL CASO")
    fc = a_fecha(f.get("FECHA DE CIERRE")) or (a_fecha(mc_raw) if isinstance(mc_raw, (datetime, date)) else None)
    mc = fc.month if fc else (None if vacio(mc_raw) else mes_num(mc_raw))
    ac = fc.year if fc else (None if vacio(ac_raw) else anio_num(ac_raw))
    if not vacio(mc_raw) and mc is None:
        adv("MES DE CIERRE DEL CASO", mc_raw, "Mes de cierre no reconocido: se deja vacío.")
    if not vacio(ac_raw) and ac is None:
        adv("AÑO DE CIERRE DEL CASO", ac_raw, "Año de cierre no reconocido: se deja vacío.")
    if mc and not ac and fecha:
        ac = fecha.year if mc >= fecha.month else fecha.year + 1
        adv("AÑO DE CIERRE DEL CASO", "", f"Año de cierre vacío: se asumió {ac}.")

    # --- Estado
    est_raw = texto(f.get("ESTADO"))
    estado = EQUIV_ESTADO.get(clave(est_raw))
    if estado is None or not est_raw:
        inferido = "Cerrado" if mc else ("En Proceso" if c["AVANCES DEL CASO"] else "Abierto")
        if est_raw:
            adv("ESTADO", est_raw, f"Estado no reconocido: se asignó '{inferido}'.")
        else:
            adv("ESTADO", "", f"Estado vacío: se asignó '{inferido}'.")
        estado = inferido
    c["ESTADO"] = estado
    registrar_cambio(n, id_txt, "ESTADO", est_raw, estado, "estado normalizado")

    if estado == "Cerrado":
        c["MES DE CIERRE DEL CASO"] = MESES[mc - 1] if mc else None
        c["AÑO DE CIERRE DEL CASO"] = ac
        if not mc:
            adv("MES DE CIERRE DEL CASO", "", "Caso cerrado sin mes de cierre (el indicador de tiempo de resolución no lo contará).")
        elif fecha and (ac, mc) < (fecha.year, fecha.month):
            adv("MES DE CIERRE DEL CASO", f"{MESES[mc - 1]} {ac}", "El cierre es anterior a la fecha de alertamiento.")
    else:
        c["MES DE CIERRE DEL CASO"], c["AÑO DE CIERRE DEL CASO"] = "-", "-"
        if mc:
            adv("MES DE CIERRE DEL CASO", f"{texto(mc_raw)} {texto(ac_raw)}",
                f"Tiene mes de cierre pero el estado es '{estado}': se dejó '-' (revise si en realidad está cerrado).")
    registrar_cambio(n, id_txt, "MES DE CIERRE DEL CASO", texto(mc_raw), texto(c["MES DE CIERRE DEL CASO"]), "mes normalizado")

    # --- Causa raíz
    raiz_raw = texto(f.get("EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ"))
    raiz = EQUIV_RAIZ.get(clave(raiz_raw))
    if raiz is None:
        adv("EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ", raiz_raw, "Respuesta no reconocida: se dejó 'En evaluación'.")
        raiz = "En evaluación"
    c["EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ"] = raiz
    registrar_cambio(n, id_txt, "EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ", raiz_raw, raiz, "respuesta normalizada")
    if estado == "Cerrado" and raiz == "En evaluación":
        adv("EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ", raiz_raw, "Caso cerrado sin indicar si se solucionó la causa raíz.")

    # --- Columnas técnicas que ya vengan en el origen (si se limpia un archivo de la app)
    for col in COLUMNAS_TECNICAS:
        v = f.get(col)
        if col in COLUMNAS_FECHA_TEXTO:
            t = a_fecha_hora_texto(v, solo_fecha=(col == "FECHA_LIMITE"))
            if t is None:
                adv(col, v, "Fecha técnica no reconocida: se dejó vacía.")
                t = ""
            c[col] = t
        elif col in ("CORREO_CREADOR", "CORREO_ANALISTA_ASIGNADO", "RESPONSABLE_ULTIMA_ACTUALIZACION"):
            c[col] = ";".join(p.strip().lower() for p in texto(v).replace(",", ";").split(";") if p.strip())
        elif col == "MOTIVO_DEVOLUCION":
            c[col] = texto(v, multilinea=True)
        else:
            c[col] = texto(v)
    return c


CASOS = [limpiar_fila(f) for f in FILAS]

# --- Duplicados exactos (todas las columnas originales iguales)
vistos, EXCLUIDAS = {}, []
for c in CASOS:
    firma = tuple(clave(c.get(col)) for col in COLUMNAS_ORIGINALES)
    if firma in vistos:
        if ELIMINAR_DUPLICADOS_EXACTOS:
            c["__excluir__"] = f"Duplicado exacto de la fila {vistos[firma]}"
        else:
            registrar_problema("ADVERTENCIA", c["__fila__"], c[COLUMNA_ID], "(fila)", "", f"Duplicado exacto de la fila {vistos[firma]}")
    else:
        vistos[firma] = c["__fila__"]

# --- Duplicados probables (misma fecha y misma descripción)
probables = {}
for c in CASOS:
    if c.get("__excluir__"):
        continue
    k = (c["__fecha__"], clave(c["DESCRIPCION DEL RIESGO"])[:80])
    if k[1] and k in probables:
        registrar_problema("ADVERTENCIA", c["__fila__"], c[COLUMNA_ID], "DESCRIPCION DEL RIESGO", "",
                           f"Posible duplicado de la fila {probables[k]} (misma fecha y descripción).")
    else:
        probables[k] = c["__fila__"]

for c in CASOS:
    if c.get("__excluir__") is None and c["__errores__"] and EXCLUIR_FILAS_CON_ERRORES:
        c["__excluir__"] = "; ".join(c["__errores__"])
    if c.get("__excluir__"):
        EXCLUIDAS.append((c["__fila__"], c[COLUMNA_ID], c["__excluir__"]))
CASOS_OK = [c for c in CASOS if not c.get("__excluir__")]

# --- IdNum únicos: se respetan los existentes; los vacíos o repetidos reciben el siguiente número
usados, REMAPEO_ID = set(), {}
siguiente = max([c[COLUMNA_ID] or 0 for c in CASOS_OK] + [0]) + 1
for c in sorted(CASOS_OK, key=lambda x: (x[COLUMNA_ID] is None, x[COLUMNA_ID] or 0, x["__fila__"])):
    original = c[COLUMNA_ID]
    if original is None or original in usados:
        c[COLUMNA_ID] = siguiente
        siguiente += 1
        registrar_cambio(c["__fila__"], original, COLUMNA_ID, original, c[COLUMNA_ID],
                         "IdNum vacío o repetido: se asignó uno nuevo")
    else:
        REMAPEO_ID[original] = c[COLUMNA_ID]
    usados.add(c[COLUMNA_ID])
CASOS_OK.sort(key=lambda x: x[COLUMNA_ID])

print(f"Filas leídas: {len(CASOS)}  ·  pasan a la app: {len(CASOS_OK)}  ·  excluidas: {len(EXCLUIDAS)}")
print(f"Correcciones registradas: {len(CAMBIOS)}  ·  problemas: "
      f"{sum(1 for p in PROBLEMAS if p[0] == 'ERROR')} errores, {sum(1 for p in PROBLEMAS if p[0] == 'ADVERTENCIA')} advertencias")
pd.DataFrame([{k: c[k] for k in [COLUMNA_ID] + COLUMNAS_ORIGINALES[:9] + ["ESTADO"]} for c in CASOS_OK]).head(20)

## 7. Correos, usuarios, catálogos y configuración
Orden para encontrar el correo de cada persona: parámetro `CORREOS` → catálogos/usuarios del archivo
de la app (si se indicó) → correos que ya traiga el origen → correo de **ejemplo** `@example.com`.

In [ ]:
def slug_correo(nombre):
    s = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode().lower()
    partes = [p for p in re.split(r"[^a-z]+", s) if p]
    return f"{partes[0]}.{partes[-1]}@example.com" if partes else "sin.nombre@example.com"


DIRECTORIO = {}     # clave(nombre) -> correo


def anotar_correo(nombre, correo):
    """Guarda el correo de una persona; un correo real nunca se reemplaza por uno de ejemplo."""
    nombre = MAPAS.get("ANALISTA ASIGANDO", {}).get(clave(nombre), texto(nombre))
    correo = ";".join(p.strip().lower() for p in texto(correo).replace(",", ";").split(";") if p.strip())
    if not nombre or "@" not in correo:
        return
    actual = DIRECTORIO.get(clave(nombre))
    if actual and "example.com" not in actual and "example.com" in correo:
        return
    DIRECTORIO[clave(nombre)] = correo


for c in CASOS_OK:      # correos que ya vengan en las columnas técnicas de los casos
    anotar_correo(c["ANALISTA ASIGANDO"], c.get("CORREO_ANALISTA_ASIGNADO"))
    anotar_correo(c.get("NOMBRE_CREADOR") or c["ANALISTA ENCARGADA UAC"], c.get("CORREO_CREADOR"))
for wb in (wb_origen, wb_app):     # el de la app tiene prioridad (se carga de último)
    for r in leer_tabla_simple(wb, "USUARIOS", COLUMNAS_USUARIOS):
        anotar_correo(r.get("NOMBRE"), r.get("CORREO"))
    for r in leer_tabla_simple(wb, "CAT_ANALISTAS_ASIGNADOS", ["NOMBRE", "CORREO", "AREA", "ACTIVO"]):
        anotar_correo(r.get("NOMBRE"), r.get("CORREO"))
for k, v in CORREOS.items():
    DIRECTORIO[clave(k)] = texto(v).lower()

CORREOS_EJEMPLO = set()


def correo_de(nombre):
    if not nombre:
        return ""
    k = clave(nombre)
    if k in DIRECTORIO and "example.com" not in DIRECTORIO[k]:
        return DIRECTORIO[k]
    CORREOS_EJEMPLO.add(nombre)
    return DIRECTORIO.get(k) or slug_correo(nombre)


# --- Columnas técnicas de cada caso (solo se completan las vacías)
AHORA = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
for c in CASOS_OK:
    creado = c["__fecha__"]
    if c["FECHA_CREACION"] and c["FECHA_CREACION"][:10] != creado.isoformat():
        registrar_problema("ADVERTENCIA", c["__fila__"], c[COLUMNA_ID], "FECHA_CREACION", c["FECHA_CREACION"],
                           f"No coincide con DIA/MES/AÑO ({creado.isoformat()}). Se conservó FECHA_CREACION.")
    defecto = {
        "FECHA_CREACION": creado.strftime("%Y-%m-%d") + " 00:00:00",
        "CORREO_CREADOR": correo_de(c["ANALISTA ENCARGADA UAC"]),
        "NOMBRE_CREADOR": c["ANALISTA ENCARGADA UAC"],
        "CORREO_ANALISTA_ASIGNADO": correo_de(c["ANALISTA ASIGANDO"]),
        "FECHA_ASIGNACION": creado.strftime("%Y-%m-%d") + " 00:00:00",
        "FECHA_LIMITE": (creado + timedelta(days=DIAS_LIMITE)).strftime("%Y-%m-%d"),
        "ESCALAMIENTO_ENVIADO": "No",
        "ULTIMA_ACTUALIZACION": AHORA,
        "RESPONSABLE_ULTIMA_ACTUALIZACION": "MIGRACION",
        "NIVEL_ESCALAMIENTO": 0,
        "TOKEN_CREACION": f"MIG-{c[COLUMNA_ID]}",
        "VERSION": 1,
        "ORIGEN_REGISTRO": "MIGRACION",
    }
    for col, v in defecto.items():
        if c.get(col) in (None, ""):
            c[col] = v
    # correo de ejemplo existente pero ahora hay uno real → se actualiza
    if "example.com" in c["CORREO_ANALISTA_ASIGNADO"] and "example.com" not in correo_de(c["ANALISTA ASIGANDO"]):
        c["CORREO_ANALISTA_ASIGNADO"] = correo_de(c["ANALISTA ASIGANDO"])
    if "example.com" in c["CORREO_CREADOR"] and "example.com" not in correo_de(c["NOMBRE_CREADOR"] or c["ANALISTA ENCARGADA UAC"]):
        c["CORREO_CREADOR"] = correo_de(c["NOMBRE_CREADOR"] or c["ANALISTA ENCARGADA UAC"])
    c["ESCALAMIENTO_ENVIADO"] = "Sí" if clave(c["ESCALAMIENTO_ENVIADO"]) in ("si", "s", "yes", "true", "1") else "No"
    c["VERSION"] = int(c["VERSION"]) if re.fullmatch(r"\d+", texto(c["VERSION"])) and int(texto(c["VERSION"])) > 0 else 1
    c["NIVEL_ESCALAMIENTO"] = int(c["NIVEL_ESCALAMIENTO"]) if re.fullmatch(r"\d+", texto(c["NIVEL_ESCALAMIENTO"])) else 0
    c["ORIGEN_REGISTRO"] = texto(c["ORIGEN_REGISTRO"]).upper() or "MIGRACION"
    if c["ESTADO"] == "Cerrado" and not c["FECHA_CIERRE"]:
        pass   # el original solo tiene mes/año de cierre; la app lo aproxima al día 15 de ese mes

# --- Catálogos: valores usados en los casos + los que ya existían en la app (con su ACTIVO)
def catalogo(hoja, campo, columnas=("NOMBRE", "ACTIVO")):
    filas = OrderedDict()
    for v in sorted({c[campo] for c in CASOS_OK if c[campo]}):
        filas[clave(v)] = {"NOMBRE": v, "ACTIVO": "Sí"}
    for wb in (wb_origen, wb_app):
        for r in leer_tabla_simple(wb, hoja, list(columnas)):
            nombre = MAPAS.get(campo, {}).get(clave(r.get("NOMBRE")), texto(r.get("NOMBRE")))
            if not nombre:
                continue
            activo = "No" if clave(r.get("ACTIVO")) == "no" else "Sí"
            filas.setdefault(clave(nombre), {"NOMBRE": nombre})["ACTIVO"] = activo
    return sorted(filas.values(), key=lambda r: clave(r["NOMBRE"]))


CAT_ANALISTAS_UAC = catalogo("CAT_ANALISTAS_UAC", "ANALISTA ENCARGADA UAC")
CAT_SPONSORS = catalogo("CAT_SPONSORS", "SPONSOR")
CAT_AREAS = catalogo("CAT_AREAS", "ÁREA ENCARGADA")
CAT_CASUALES = catalogo("CAT_CASUALES", "CASUAL")
CAT_RESP = catalogo("CAT_RESP_OPERATIVO", "RESPONSABLE OPERATIVO DEL ALERTAMIENTO")

subc = OrderedDict()
for c in sorted(CASOS_OK, key=lambda x: (clave(x["CASUAL"]), clave(x["SUBCASUAL"]))):
    if c["CASUAL"] and c["SUBCASUAL"]:
        subc[(clave(c["CASUAL"]), clave(c["SUBCASUAL"]))] = {"CASUAL": c["CASUAL"], "NOMBRE": c["SUBCASUAL"], "ACTIVO": "Sí"}
for wb in (wb_origen, wb_app):
    for r in leer_tabla_simple(wb, "CAT_SUBCASUALES", ["CASUAL", "NOMBRE", "ACTIVO"]):
        ca = MAPAS.get("CASUAL", {}).get(clave(r.get("CASUAL")), texto(r.get("CASUAL")))
        no = MAPAS.get("SUBCASUAL", {}).get(clave(r.get("NOMBRE")), texto(r.get("NOMBRE")))
        if ca and no:
            subc.setdefault((clave(ca), clave(no)), {"CASUAL": ca, "NOMBRE": no})["ACTIVO"] = "No" if clave(r.get("ACTIVO")) == "no" else "Sí"
CAT_SUBCASUALES = sorted(subc.values(), key=lambda r: (clave(r["CASUAL"]), clave(r["NOMBRE"])))

asig = OrderedDict()
for c in CASOS_OK:
    if c["ANALISTA ASIGANDO"]:
        asig[(clave(c["ANALISTA ASIGANDO"]), clave(c["ÁREA ENCARGADA"]))] = {
            "NOMBRE": c["ANALISTA ASIGANDO"], "CORREO": correo_de(c["ANALISTA ASIGANDO"]),
            "AREA": c["ÁREA ENCARGADA"], "ACTIVO": "Sí"}
for wb in (wb_origen, wb_app):
    for r in leer_tabla_simple(wb, "CAT_ANALISTAS_ASIGNADOS", ["NOMBRE", "CORREO", "AREA", "ACTIVO"]):
        no = MAPAS.get("ANALISTA ASIGANDO", {}).get(clave(r.get("NOMBRE")), texto(r.get("NOMBRE")))
        ar = texto(r.get("AREA"))
        ar = "TODAS" if clave(ar) == "todas" else MAPAS.get("ÁREA ENCARGADA", {}).get(clave(ar), ar)
        if no:
            fila = asig.setdefault((clave(no), clave(ar)), {"NOMBRE": no, "AREA": ar})
            fila["CORREO"] = correo_de(no)
            fila["ACTIVO"] = "No" if clave(r.get("ACTIVO")) == "no" else "Sí"
CAT_ANALISTAS_ASIGNADOS = sorted(asig.values(), key=lambda r: (clave(r["AREA"]), clave(r["NOMBRE"])))

# --- Usuarios: los de la app (o del origen) + quien falte de los casos
USUARIOS = OrderedDict()
hoy_txt = date.today().strftime("%Y-%m-%d")
# Si se indicó el archivo de la app, sus USUARIOS mandan; si no, los que traiga el origen (si trae).
for wb in ([wb_app] if wb_app is not None else [wb_origen]):
    for r in leer_tabla_simple(wb, "USUARIOS", COLUMNAS_USUARIOS):
        correo = texto(r.get("CORREO")).lower()
        if "@" not in correo:
            continue
        rol = texto(r.get("ROL")).upper()
        USUARIOS[correo] = {
            "CORREO": correo, "NOMBRE": texto(r.get("NOMBRE")),
            "ROL": rol if rol in ROLES else "SOLICITANTE",
            "AREA": texto(r.get("AREA")), "ACTIVO": "No" if clave(r.get("ACTIVO")) == "no" else "Sí",
            "FECHA_ALTA": a_fecha_hora_texto(r.get("FECHA_ALTA"), solo_fecha=True) or hoy_txt,
            "OBSERVACIONES": texto(r.get("OBSERVACIONES")),
        }
if ADMIN_CORREO:
    a = ADMIN_CORREO.strip().lower()
    previo = USUARIOS.pop(a, {})
    USUARIOS = OrderedDict([(a, {**previo, "CORREO": a, "NOMBRE": ADMIN_NOMBRE or previo.get("NOMBRE") or a,
                                  "ROL": "ADMINISTRADOR", "AREA": previo.get("AREA") or "UAC", "ACTIVO": "Sí",
                                  "FECHA_ALTA": previo.get("FECHA_ALTA") or hoy_txt,
                                  "OBSERVACIONES": previo.get("OBSERVACIONES", "")})] + list(USUARIOS.items()))
if not any(u["ROL"] == "ADMINISTRADOR" for u in USUARIOS.values()):
    USUARIOS = OrderedDict([("admin.mesa@example.com", {
        "CORREO": "admin.mesa@example.com", "NOMBRE": "ADMINISTRADOR DEMO", "ROL": "ADMINISTRADOR", "AREA": "UAC",
        "ACTIVO": "Sí", "FECHA_ALTA": hoy_txt, "OBSERVACIONES": "CORREO DE EJEMPLO – reemplazar"})] + list(USUARIOS.items()))

nombres_existentes = {clave(u["NOMBRE"]) for u in USUARIOS.values()}
areas_por_analista = {}
for r in CAT_ANALISTAS_ASIGNADOS:
    areas_por_analista.setdefault(r["NOMBRE"], set()).add(r["AREA"])
for nombre, rol, area in ([(r["NOMBRE"], "SOLICITANTE", "UAC") for r in CAT_ANALISTAS_UAC] +
                          [(n, "ESPECIALISTA", ", ".join(sorted(a))) for n, a in sorted(areas_por_analista.items())]):
    if clave(nombre) in nombres_existentes:
        continue
    correo = correo_de(nombre)
    if correo in USUARIOS:
        continue
    USUARIOS[correo] = {"CORREO": correo, "NOMBRE": nombre, "ROL": rol, "AREA": area, "ACTIVO": "Sí",
                        "FECHA_ALTA": hoy_txt,
                        "OBSERVACIONES": "CORREO DE EJEMPLO – reemplazar" if "example.com" in correo else ""}
    nombres_existentes.add(clave(nombre))

# --- Configuración
CONFIG = OrderedDict((k, [v, d]) for k, v, d in CONFIG_DEFECTO)
for wb in (wb_origen, wb_app):
    for r in leer_tabla_simple(wb, "CONFIG", ["CLAVE", "VALOR", "DESCRIPCION"]):
        k = texto(r.get("CLAVE")).upper()
        if k:
            CONFIG.setdefault(k, ["", texto(r.get("DESCRIPCION"))])[0] = texto(r.get("VALOR"))
CONFIG["DIAS_LIMITE"][0] = str(DIAS_LIMITE)     # coherente con las FECHA_LIMITE calculadas arriba
CONFIG["ID_INICIAL"][0] = str(max([c[COLUMNA_ID] for c in CASOS_OK] + [0]) + 1)

for nombre in sorted(CORREOS_EJEMPLO):
    registrar_problema("ADVERTENCIA", "", "", "CORREO", nombre,
                       "Sin correo real: se usó uno @example.com. Agréguelo en CORREOS o en USUARIOS / CAT_ANALISTAS_ASIGNADOS.")
print(f"Usuarios: {len(USUARIOS)}  ·  analistas asignados: {len(CAT_ANALISTAS_ASIGNADOS)}  ·  "
      f"áreas: {len(CAT_AREAS)}  ·  causales: {len(CAT_CASUALES)}  ·  subcausales: {len(CAT_SUBCASUALES)}")
print(f"Personas con correo de EJEMPLO: {len(CORREOS_EJEMPLO)}")

## 8. Historial
Si el origen ya trae la hoja `HISTORIAL_CASOS` (por ejemplo, al limpiar un archivo que ya usa la app)
se conserva. A cada caso sin historial se le agregan los eventos de migración (creación y estado).

In [ ]:
HISTORIAL = []
ids_ok = {c[COLUMNA_ID] for c in CASOS_OK}
for r in leer_tabla_simple(wb_origen, "HISTORIAL_CASOS", COLUMNAS_HISTORIAL):
    id_txt = texto(r.get("ID_CASO"))
    id_viejo = int(id_txt) if re.fullmatch(r"\d+", id_txt) else None
    id_nuevo = REMAPEO_ID.get(id_viejo)
    if id_nuevo not in ids_ok:
        registrar_problema("ADVERTENCIA", "", id_txt, "HISTORIAL_CASOS", texto(r.get("TIPO_EVENTO")),
                           "Evento de historial de un caso que no pasó a la app: se descartó.")
        continue
    h = {col: texto(r.get(col), multilinea=(col == "DETALLE")) for col in COLUMNAS_HISTORIAL}
    h["ID_CASO"] = id_nuevo
    h["FECHA"] = a_fecha_hora_texto(r.get("FECHA")) or ""
    h["USUARIO_CORREO"] = h["USUARIO_CORREO"].lower()
    HISTORIAL.append(h)

con_creacion = {h["ID_CASO"] for h in HISTORIAL if h["TIPO_EVENTO"].upper() == "CREACION"}
nuevos = []
for c in CASOS_OK:
    if c[COLUMNA_ID] in con_creacion:
        continue
    nuevos.append({"ID_CASO": c[COLUMNA_ID], "FECHA": c["FECHA_CREACION"], "TIPO_EVENTO": "CREACION",
                   "ESTADO_ANTERIOR": "", "ESTADO_NUEVO": "Abierto", "DETALLE": "[MIGRACION] Caso registrado.",
                   "VALOR_ANTERIOR": "", "VALOR_NUEVO": c["ANALISTA ASIGANDO"],
                   "USUARIO_CORREO": c["CORREO_CREADOR"], "USUARIO_NOMBRE": c["NOMBRE_CREADOR"], "ORIGEN": "MIGRACION"})
    if c["ESTADO"] != "Abierto":
        nuevos.append({"ID_CASO": c[COLUMNA_ID], "FECHA": AHORA, "TIPO_EVENTO": "CAMBIO_ESTADO",
                       "ESTADO_ANTERIOR": "", "ESTADO_NUEVO": c["ESTADO"],
                       "DETALLE": "[MIGRACION] Estado, avances y plan importados del archivo original.",
                       "VALOR_ANTERIOR": "", "VALOR_NUEVO": c["ESTADO"], "USUARIO_CORREO": "sistema",
                       "USUARIO_NOMBRE": "MIGRACION", "ORIGEN": "notebook_limpieza"})
usados_h = {h["ID_HISTORIAL"] for h in HISTORIAL if h["ID_HISTORIAL"]}
i = 0
for h in HISTORIAL + nuevos:
    if not h.get("ID_HISTORIAL") or (h in nuevos and h["ID_HISTORIAL"] in usados_h):
        i += 1
        while f"H-MIGRACION-{i:04d}" in usados_h:
            i += 1
        h["ID_HISTORIAL"] = f"H-MIGRACION-{i:04d}"
        usados_h.add(h["ID_HISTORIAL"])
HISTORIAL = sorted(HISTORIAL + nuevos, key=lambda h: (h["ID_CASO"], h["FECHA"]))

NOTIFICACIONES = [{k: texto(r.get(k)) for k in COLS_NOTIF} for r in leer_tabla_simple(wb_origen, "NOTIFICACIONES", COLS_NOTIF)]
LOG_ERRORES = [{k: texto(r.get(k)) for k in COLS_LOG} for r in leer_tabla_simple(wb_origen, "LOG_ERRORES", COLS_LOG)]
print(f"Eventos de historial: {len(HISTORIAL)} ({len(nuevos)} nuevos de migración)")

## 9. Escribir el Excel para la app
Mismas hojas, tablas (`TablaCasos`, `tblHistorial`, `tblUsuarios`, `tblCat…`, `tblConfig`…) y columnas
que el archivo actual de la app. Las fechas técnicas se guardan como texto `aaaa-mm-dd hh:mm:ss`.

In [ ]:
FUENTE, MORADO, GRIS_TEC = "Arial", "3B0A8F", "4B5563"


def estilo_encabezado(ws, ncols, color=MORADO, desde=1):
    for col in range(desde, ncols + 1):
        cell = ws.cell(row=1, column=col)
        cell.font = Font(name=FUENTE, bold=True, color="FFFFFF", size=10)
        cell.fill = PatternFill("solid", fgColor=color)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 42


def crear_tabla(wb, hoja, nombre_tabla, columnas, filas, anchos=None, columnas_texto=(), estilo="TableStyleMedium2"):
    ws = wb.create_sheet(hoja)
    ws.append(columnas)
    for fila in filas:
        valores = [fila.get(c, "") for c in columnas] if isinstance(fila, dict) else list(fila)
        ws.append([None if v == "" else v for v in valores])
    estilo_encabezado(ws, len(columnas))
    n = max(ws.max_row, 2)
    tabla = Table(displayName=nombre_tabla, ref=f"A1:{get_column_letter(len(columnas))}{n}")
    tabla.tableStyleInfo = TableStyleInfo(name=estilo, showRowStripes=True)
    ws.add_table(tabla)
    for fila_celdas in ws.iter_rows(min_row=2, max_row=n):
        for cell in fila_celdas:
            cell.font = Font(name=FUENTE, size=10)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
    for i, col in enumerate(columnas, 1):
        letra = get_column_letter(i)
        ws.column_dimensions[letra].width = (anchos or {}).get(col, max(14, min(len(col) + 4, 40)))
        if col in columnas_texto:
            for r in range(2, n + 1):
                ws[f"{letra}{r}"].number_format = "@"
    ws.freeze_panes = "B2"
    return ws


def lista_validacion(ws, letra, valores, hasta):
    dv = DataValidation(type="list", formula1='"' + ",".join(valores) + '"', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f"{letra}2:{letra}{hasta}")


wb = Workbook()
wb.remove(wb.active)

# ---- Hoja 1 / TablaCasos
filas_casos = []
for c in CASOS_OK:
    filas_casos.append({col: c.get(col, "") for col in COLUMNAS_CASOS})
ws = crear_tabla(wb, HOJA_CASOS, TABLA_CASOS, COLUMNAS_CASOS, filas_casos,
                 {"IdNum": 8, "DESCRIPCION DEL RIESGO": 50, "AVANCES DEL CASO": 45, "PLAN DE ACCION": 45,
                  "SUBCASUAL": 36, "MOTIVO_DEVOLUCION": 40, "CORREO_ANALISTA_ASIGNADO": 34},
                 COLUMNAS_FECHA_TEXTO + ["ALERTAS_ENVIADAS", "TOKEN_CREACION"], estilo="TableStyleMedium9")
estilo_encabezado(ws, len(COLUMNAS_ORIGINALES) + 1)
estilo_encabezado(ws, len(COLUMNAS_CASOS), GRIS_TEC, desde=len(COLUMNAS_ORIGINALES) + 2)
for i, col in enumerate(COLUMNAS_CASOS, 1):
    if col not in ("IdNum", "DESCRIPCION DEL RIESGO", "AVANCES DEL CASO", "PLAN DE ACCION", "SUBCASUAL",
                   "MOTIVO_DEVOLUCION", "CORREO_ANALISTA_ASIGNADO"):
        ws.column_dimensions[get_column_letter(i)].width = max(12, min(len(col) + 3, 30))
ws.cell(row=1, column=COLUMNAS_CASOS.index("ANALISTA ASIGANDO") + 1).comment = Comment(
    "Nombre original conservado ('ASIGANDO'). No corregir: la app y los flujos dependen de él.", "Mesa de Servicio")
ws.cell(row=1, column=COLUMNAS_CASOS.index("CASUAL") + 1).comment = Comment(
    "Nombre original conservado ('CASUAL' = causal). Igual SUBCASUAL. No corregir.", "Mesa de Servicio")
ws.cell(row=1, column=1).comment = Comment("IdNum = identificador único del caso.", "Mesa de Servicio")
ws.cell(row=1, column=len(COLUMNAS_ORIGINALES) + 2).comment = Comment(
    "Desde aquí: columnas técnicas escritas por la app y los flujos.", "Mesa de Servicio")
hasta = len(CASOS_OK) + 1001
lista_validacion(ws, get_column_letter(COLUMNAS_CASOS.index("ESTADO") + 1), ESTADOS_VALIDOS, hasta)
lista_validacion(ws, get_column_letter(COLUMNAS_CASOS.index("EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ") + 1), VALORES_RAIZ, hasta)
lista_validacion(ws, get_column_letter(COLUMNAS_CASOS.index("ESCALAMIENTO_ENVIADO") + 1), ["Sí", "No"], hasta)

# ---- Resto de hojas
crear_tabla(wb, "HISTORIAL_CASOS", "tblHistorial", COLUMNAS_HISTORIAL, HISTORIAL,
            {"DETALLE": 60, "VALOR_NUEVO": 40, "ID_HISTORIAL": 38}, ["FECHA", "ID_HISTORIAL"])
wsu = crear_tabla(wb, "USUARIOS", "tblUsuarios", COLUMNAS_USUARIOS, list(USUARIOS.values()),
                  {"CORREO": 34, "NOMBRE": 26, "AREA": 40, "OBSERVACIONES": 34}, ["FECHA_ALTA"])
lista_validacion(wsu, "C", ROLES, 500)
lista_validacion(wsu, "E", ["Sí", "No"], 500)
crear_tabla(wb, "CAT_ANALISTAS_UAC", "tblCatAnalistasUAC", ["NOMBRE", "ACTIVO"], CAT_ANALISTAS_UAC, {"NOMBRE": 30})
crear_tabla(wb, "CAT_SPONSORS", "tblCatSponsors", ["NOMBRE", "ACTIVO"], CAT_SPONSORS, {"NOMBRE": 30})
crear_tabla(wb, "CAT_AREAS", "tblCatAreas", ["NOMBRE", "ACTIVO"], CAT_AREAS, {"NOMBRE": 34})
wsa = crear_tabla(wb, "CAT_ANALISTAS_ASIGNADOS", "tblCatAnalistasAsignados", ["NOMBRE", "CORREO", "AREA", "ACTIVO"],
                  CAT_ANALISTAS_ASIGNADOS, {"NOMBRE": 26, "CORREO": 40, "AREA": 34})
wsa["B1"].comment = Comment("Uno o varios correos separados por ';'. Los example.com son de EJEMPLO.", "Mesa de Servicio")
wsa["C1"].comment = Comment("Área que atiende. Use TODAS para cualquier área.", "Mesa de Servicio")
crear_tabla(wb, "CAT_CASUALES", "tblCatCausales", ["NOMBRE", "ACTIVO"], CAT_CASUALES, {"NOMBRE": 30})
crear_tabla(wb, "CAT_SUBCASUALES", "tblCatSubcausales", ["CASUAL", "NOMBRE", "ACTIVO"], CAT_SUBCASUALES,
            {"CASUAL": 28, "NOMBRE": 62})
crear_tabla(wb, "CAT_RESP_OPERATIVO", "tblCatRespOperativo", ["NOMBRE", "ACTIVO"], CAT_RESP, {"NOMBRE": 30})
crear_tabla(wb, "CAT_ESTADOS", "tblCatEstados", ["CODIGO", "ETIQUETA", "ORDEN", "COLOR_HEX", "ES_FINAL"], ESTADOS,
            {"ETIQUETA": 26})
crear_tabla(wb, "EQUIVALENCIAS_ESTADO", "tblEquivalenciasEstado", ["ESTADO_ORIGINAL", "ESTADO_NUEVO"],
            EQUIVALENCIAS_ESTADO_HOJA, {"ESTADO_ORIGINAL": 24, "ESTADO_NUEVO": 18})
for nombre_cat in ("CAT_ANALISTAS_UAC", "CAT_SPONSORS", "CAT_AREAS", "CAT_CASUALES", "CAT_RESP_OPERATIVO"):
    lista_validacion(wb[nombre_cat], "B", ["Sí", "No"], 500)
lista_validacion(wb["CAT_SUBCASUALES"], "C", ["Sí", "No"], 500)
lista_validacion(wsa, "D", ["Sí", "No"], 500)

wsc = crear_tabla(wb, "CONFIG", "tblConfig", ["CLAVE", "VALOR", "DESCRIPCION"],
                  [(k, v, d) for k, (v, d) in CONFIG.items()],
                  {"CLAVE": 34, "VALOR": 46, "DESCRIPCION": 90}, ["VALOR"], estilo="TableStyleMedium4")
for r in range(2, wsc.max_row + 1):
    if str(wsc[f"B{r}"].value or "").startswith("["):
        wsc[f"B{r}"].fill = PatternFill("solid", fgColor="FFF4B3")
crear_tabla(wb, "PLANTILLAS_CORREO", "tblPlantillas", ["CODIGO", "ARCHIVO", "ASUNTO", "ACTIVA"], PLANTILLAS,
            {"CODIGO": 24, "ARCHIVO": 34, "ASUNTO": 80})
crear_tabla(wb, "NOTIFICACIONES", "tblNotificaciones", COLS_NOTIF, NOTIFICACIONES,
            {"CLAVE_UNICA": 36, "DESTINATARIOS": 40, "ASUNTO": 60, "DETALLE": 50}, ["FECHA_ENVIO"])
crear_tabla(wb, "LOG_ERRORES", "tblLogErrores", COLS_LOG, LOG_ERRORES,
            {"MENSAJE": 50, "DETALLE": 70, "ID_EJECUCION": 36}, ["FECHA"])

# ---- LEEME
leeme = wb.create_sheet("LEEME", 0)
lineas = [
    ("SEGUIMIENTO DE ALERTAMIENTOS Y GESTIÓN DE RIESGOS – DATOS DE LA MESA DE SERVICIO", True),
    (f"Generado el {AHORA} por el notebook de limpieza a partir de '{RUTA_ORIGEN.name}' ({len(CASOS_OK)} casos).", False),
    ("", False), ("REGLAS DE USO", True),
    ("1. No renombre hojas, tablas ni columnas: la app depende de ellos (incluidos 'CASUAL', 'SUBCASUAL' y 'ANALISTA ASIGANDO').", False),
    ("2. Columnas de encabezado morado: datos del caso. Encabezado gris: columnas técnicas que escriben la app y los flujos.", False),
    ("3. MES y MES DE CIERRE como nombre ('Enero'); '-' en mes/año de cierre de casos no cerrados.", False),
    ("4. Fechas técnicas como TEXTO 'aaaa-mm-dd hh:mm:ss' (hora de Bogotá).", False),
    ("5. No elimine filas de HISTORIAL_CASOS, NOTIFICACIONES ni LOG_ERRORES.", False),
    ("6. Hojas CAT_*: listas desplegables de la app. Para retirar un valor ponga ACTIVO = No (no lo borre).", False),
    ("7. Los correos @example.com son de EJEMPLO: reemplácelos por los reales en USUARIOS y CAT_ANALISTAS_ASIGNADOS.", False),
    ("8. Complete en CONFIG los valores [ENTRE_CORCHETES] (resaltados en amarillo).", False),
    ("", False), ("CÓMO SE MIGRARON LOS CASOS", True),
    (f"• FECHA_CREACION = DIA/MES/AÑO; FECHA_LIMITE = +{DIAS_LIMITE} días. FECHA_CIERRE queda vacía cuando el origen solo trae mes/año de cierre.", False),
    ("• Las correcciones hechas durante la limpieza están en el archivo Reporte_limpieza_datos.xlsx.", False),
]
for i, (t, negrita) in enumerate(lineas, 1):
    celda = leeme.cell(row=i, column=1, value=t)
    celda.font = Font(name=FUENTE, bold=negrita, size=12 if negrita else 10, color=MORADO if negrita else "222222")
    celda.alignment = Alignment(wrap_text=True, vertical="top")
leeme.column_dimensions["A"].width = 150
for wsx in wb.worksheets:
    wsx.sheet_properties.tabColor = MORADO if wsx.title in (HOJA_CASOS, "HISTORIAL_CASOS") else (
        "1E7D3A" if wsx.title.startswith("CAT_") else "6B7280")
wb.active = 1
wb.save(RUTA_SALIDA)
print("✅ Archivo para la app:", RUTA_SALIDA.resolve())

## 10. Reporte de limpieza
Revise sobre todo las hojas **PROBLEMAS** (errores y advertencias) y **SUGERENCIAS** (posibles nombres
repetidos con otra escritura). Si quiere aplicar una sugerencia, agréguela en `CORRECCIONES` y vuelva a ejecutar.

In [ ]:
df_resumen = pd.DataFrame([
    ("Archivo de origen", RUTA_ORIGEN.name), ("Hoja leída", HOJA_ORIGEN),
    ("Filas con datos en el origen", len(CASOS)), ("Casos que pasan a la app", len(CASOS_OK)),
    ("Filas excluidas", len(EXCLUIDAS)), ("Correcciones aplicadas", len(CAMBIOS)),
    ("Valores unificados", len(df_unif)), ("Sugerencias para revisar", len(df_sug)),
    ("Errores", sum(1 for p in PROBLEMAS if p[0] == "ERROR")),
    ("Advertencias", sum(1 for p in PROBLEMAS if p[0] == "ADVERTENCIA")),
    ("Personas con correo de ejemplo", len(CORREOS_EJEMPLO)),
    ("Siguiente IdNum", CONFIG["ID_INICIAL"][0]),
], columns=["INDICADOR", "VALOR"])
df_cambios = pd.DataFrame(CAMBIOS, columns=["FILA_ORIGEN", "IDNUM_ORIGEN", "COLUMNA", "ANTES", "DESPUES", "REGLA"])
df_problemas = pd.DataFrame(PROBLEMAS, columns=["NIVEL", "FILA_ORIGEN", "IDNUM", "COLUMNA", "VALOR", "MENSAJE"])
df_excluidas = pd.DataFrame(EXCLUIDAS, columns=["FILA_ORIGEN", "IDNUM", "MOTIVO"])
df_columnas = pd.DataFrame([(e, MAPA_COLUMNAS.get(e, "(se ignora)")) for e in ENCABEZADOS if e],
                           columns=["COLUMNA_ORIGEN", "COLUMNA_APP"])

with pd.ExcelWriter(RUTA_REPORTE, engine="openpyxl") as xw:
    for nombre, df in [("RESUMEN", df_resumen), ("PROBLEMAS", df_problemas), ("SUGERENCIAS", df_sug),
                       ("UNIFICACIONES", df_unif), ("CAMBIOS", df_cambios), ("EXCLUIDAS", df_excluidas),
                       ("COLUMNAS", df_columnas)]:
        (df if len(df) else pd.DataFrame({"(sin registros)": []})).to_excel(xw, sheet_name=nombre, index=False)
        hoja = xw.sheets[nombre]
        for i, col in enumerate(hoja.iter_cols(min_row=1, max_row=min(hoja.max_row, 200)), 1):
            ancho = max(len(str(c.value or "")) for c in col)
            hoja.column_dimensions[get_column_letter(i)].width = min(max(12, ancho + 2), 70)
            hoja.cell(row=1, column=i).font = Font(bold=True, color="FFFFFF")
            hoja.cell(row=1, column=i).fill = PatternFill("solid", fgColor=MORADO)
        hoja.freeze_panes = "A2"
print("✅ Reporte:", RUTA_REPORTE.resolve())
display(df_resumen)
display(df_problemas) if len(df_problemas) else print("Sin problemas detectados.")

## 11. Verificación final
Vuelve a abrir el archivo generado y comprueba que tiene exactamente lo que la app necesita.

In [ ]:
v = load_workbook(RUTA_SALIDA)
esperado = {
    HOJA_CASOS: (TABLA_CASOS, COLUMNAS_CASOS), "HISTORIAL_CASOS": ("tblHistorial", COLUMNAS_HISTORIAL),
    "USUARIOS": ("tblUsuarios", COLUMNAS_USUARIOS),
    "CAT_ANALISTAS_UAC": ("tblCatAnalistasUAC", ["NOMBRE", "ACTIVO"]),
    "CAT_SPONSORS": ("tblCatSponsors", ["NOMBRE", "ACTIVO"]), "CAT_AREAS": ("tblCatAreas", ["NOMBRE", "ACTIVO"]),
    "CAT_ANALISTAS_ASIGNADOS": ("tblCatAnalistasAsignados", ["NOMBRE", "CORREO", "AREA", "ACTIVO"]),
    "CAT_CASUALES": ("tblCatCausales", ["NOMBRE", "ACTIVO"]),
    "CAT_SUBCASUALES": ("tblCatSubcausales", ["CASUAL", "NOMBRE", "ACTIVO"]),
    "CAT_RESP_OPERATIVO": ("tblCatRespOperativo", ["NOMBRE", "ACTIVO"]),
    "CONFIG": ("tblConfig", ["CLAVE", "VALOR", "DESCRIPCION"]),
    "PLANTILLAS_CORREO": ("tblPlantillas", ["CODIGO", "ARCHIVO", "ASUNTO", "ACTIVA"]),
    "NOTIFICACIONES": ("tblNotificaciones", COLS_NOTIF), "LOG_ERRORES": ("tblLogErrores", COLS_LOG),
}
fallas = []
for hoja, (tabla, columnas) in esperado.items():
    if hoja not in v.sheetnames:
        fallas.append(f"Falta la hoja {hoja}")
        continue
    wsv = v[hoja]
    if tabla not in wsv.tables:
        fallas.append(f"Falta la tabla {tabla} en {hoja}")
    encabezados = [c.value for c in wsv[1]][:len(columnas)]
    if encabezados != columnas:
        fallas.append(f"Encabezados distintos en {hoja}")

casos_v = pd.DataFrame(list(v[HOJA_CASOS].iter_rows(min_row=2, values_only=True)), columns=COLUMNAS_CASOS)
casos_v = casos_v[casos_v[COLUMNA_ID].notna()]
if casos_v[COLUMNA_ID].duplicated().any():
    fallas.append("IdNum repetidos")
if not casos_v["ESTADO"].isin(ESTADOS_VALIDOS).all():
    fallas.append("Hay estados fuera de la lista de la app")
if not casos_v["MES"].isin(MESES).all():
    fallas.append("Hay meses no válidos")
if not casos_v["EL PLAN DE ACCION SOLUCIONÓ EL PROBLEMA DE RAIZ"].isin(VALORES_RAIZ).all():
    fallas.append("Hay respuestas de causa raíz no válidas")
if casos_v["FECHA_LIMITE"].map(lambda x: a_fecha(x) is None).any():
    fallas.append("Hay FECHA_LIMITE vacías o inválidas")
areas_cat = {r[0] for r in v["CAT_AREAS"].iter_rows(min_row=2, values_only=True) if r[0]}
if not set(casos_v["ÁREA ENCARGADA"].dropna()) <= areas_cat:
    fallas.append("Hay áreas en casos que no están en CAT_AREAS")
pares = {(r[0], r[1]) for r in v["CAT_SUBCASUALES"].iter_rows(min_row=2, values_only=True) if r[0]}
if not {(a, b) for a, b in zip(casos_v["CASUAL"], casos_v["SUBCASUAL"]) if a and b} <= pares:
    fallas.append("Hay causal/subcausal en casos que no están en CAT_SUBCASUALES")
ids_hist = {r[1] for r in v["HISTORIAL_CASOS"].iter_rows(min_row=2, values_only=True) if r[0]}
if not ids_hist <= set(casos_v[COLUMNA_ID]):
    fallas.append("Hay historial de casos que no existen")
if not any(r[2] == "ADMINISTRADOR" and r[4] == "Sí" for r in v["USUARIOS"].iter_rows(min_row=2, values_only=True)):
    fallas.append("No hay ningún ADMINISTRADOR activo en USUARIOS")

if fallas:
    print("❌ Revise:", *fallas, sep="\n  - ")
else:
    print(f"✅ Verificación correcta: {len(casos_v)} casos, {len(ids_hist)} casos con historial, "
          f"{v['USUARIOS'].max_row - 1} usuarios. El archivo está listo para la app.")

## 12. Cómo poner el archivo en uso
1. Cierre la app en el navegador (Studio y reproducción).
2. En OneDrive, abra la carpeta donde está el Excel de la app y **suba el archivo generado con el mismo
   nombre**, eligiendo *Reemplazar* (así conserva el mismo archivo y la app no pierde la conexión).
3. En Power Apps Studio: *Datos* → en cada tabla `…` → **Actualizar**. Luego pruebe la app.

> Si prefiere no reemplazar, puede conectar la app al archivo nuevo, pero tendría que volver a agregar
> todas las tablas con los mismos nombres.

**Antes de reemplazar:** si en la app ya se crearon casos que no están en el Excel original, descargue
primero el Excel actual de la app y úselo como `RUTA_ORIGEN` (el notebook también limpia ese formato y
conserva su historial) o como `RUTA_APP_ACTUAL` para no perder usuarios ni configuración.